# 项目-航空AI助手

现在，我们将汇集我们所学到的知识，为航空公司打造人工智能客户支持助理

In [1]:
# 进口

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# 初始化

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

   
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")
    
    
MODEL = "gpt-4.1-mini"

DB = "prices.db"


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-


In [3]:
# 连接到 OpenAI 客户端库
# 围绕 HTTP 端点调用的薄包装器

openai = OpenAI()

# 对于 Gemini、DeepSeek 和 Groq，我们可以使用 OpenAI python 客户端
# 因为 Google 和 DeepSeek 拥有与 OpenAI 兼容的端点
# OpenAI 允许您更改 base_url


openai_client = OpenAI(api_key=openai_api_key)
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
gr.ChatInterface(fn=chat, type="messages").launch()

In [4]:
# 一些用于处理图像的导入

import base64
from io import BytesIO
from PIL import Image

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def artist(city):
    image_response = openai.images.generate(
        model="gpt-image-1.5",
        prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
        output_format="png",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


In [ ]:
# 使用 Nano Banana API

# def Artist_nanobanana(城市):
# 如果不是 google_api_key:
# raise ValueError(“Nano Banana 需要 GOOGLE_API_KEY”)

# 导入 urllib.request

# Prompt = f“代表在 {city} 度假的图像，以充满活力的波普艺术风格展示旅游景点和 {city} 的一切独特之处”
# 有效负载={
# “内容”：[{“部分”：[{“文本”：提示}]}]，
# " GenerationConfig": {"responseModalities": ["TEXT", "IMAGE"]},
#     }

# url =“https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-image:generateContent”
# 请求 = urllib.request.Request(
# 网址，
# 数据=json.dumps(payload).encode("utf-8"),
# headers={"Content-Type": "application/json", "x-goog-api-key": google_api_key},
# 方法=“POST”，
#     )

# 以 urllib.request.urlopen(req) 作为响应：
# 结果 = json.loads(resp.read().decode("utf-8"))

# 对于 result.get("candidates", []) 中的候选人：
# parts = Candidate.get("内容", {}).get("部分", [])
# 对于部分中的部分：
# inline = part.get("inlineData")
# 如果内联和 inline.get("data"):
# image_data = base64.b64decode(内联[“数据”])
# 返回 Image.open(BytesIO(image_data))

# raise ValueError(f"Nano Banana 没有返回图像：{result}")

# image = Artist_nanobanana("旧金山市")
# 显示（图像）


In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
image = artist("San Francisco City")
display(image)

In [7]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [ ]:
# 回调（以及上面的 chat() 函数）

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# 用户界面定义

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# 将事件连接到回调

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))